In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
cxr_pt = "/gpfs/scratch/as16583/mimic/cxr_adm.csv"
ecg_pt = "/gpfs/scratch/as16583/mimic/ecg_adm.csv"

cxr_df = pd.read_csv(cxr_pt)[['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'discharge_location', 'hospital_expire_flag', 'dod', 'cxr_dicom_id', 'cxr_study_id', 'cxr_StudyDateTime', 'cxr_path']]
ecg_df = pd.read_csv(ecg_pt)[['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'discharge_location', 'hospital_expire_flag', 'dod', 'ecg_study_id', 'ecg_time', 'ecg_path']]

df = pd.merge(cxr_df, ecg_df, on=['subject_id', 'hadm_id', 'admittime', 'dischtime', 'deathtime', 'discharge_location', 'hospital_expire_flag', 'dod'], how='outer')

df['admittime'] = pd.to_datetime(df['admittime'])
df['dischtime'] = pd.to_datetime(df['dischtime'])
df['deathtime'] = pd.to_datetime(df['deathtime'])
df['dod'] = pd.to_datetime(df['dod'])
df['cxr_StudyDateTime'] = pd.to_datetime(df['cxr_StudyDateTime'])
df['ecg_time'] = pd.to_datetime(df['ecg_time'])

In [93]:
print("number of cxrs: ", len(cxr_df))
print("number of ecgs: ", len(ecg_df))
assert cxr_df.hadm_id.nunique() == len(cxr_df)
assert ecg_df.hadm_id.nunique() == len(ecg_df)

print("number of admissions in merged df: ", len(df))
print("number of admissions with both cxr and ecg: ", len(df[(~df['cxr_path'].isna()) & (~df['ecg_path'].isna())]))
print("number of admissions with only cxr: ", len(df[(~df['cxr_path'].isna()) & (df['ecg_path'].isna())]))
print("number of admissions with only ecg: ", len(df[(df['cxr_path'].isna()) & (~df['ecg_path'].isna())]))

number of cxrs:  70297
number of ecgs:  204652
number of admissions in merged df:  220965
number of admissions with both cxr and ecg:  53984
number of admissions with only cxr:  16313
number of admissions with only ecg:  150668


### length of stay

`admittime` provides the date and time the patient was admitted to the hospital, while `dischtime` provides the date and time the patient was discharged from the hospital.

We first create a column called `length_of_stay_days`. An LOS of 0 days indicates a stay that's less than 24 hours. An LOS of 1 day indicates a stay that's between 24 and 48 hours.

According to the MIMIC documentation, "Organ donor accounts are sometimes created for patients who died in the hospital. These are distinct hospital admissions with very short, sometimes negative lengths of stay. Furthermore, their `deathtime` is frequently the same as the earlier patient admission’s `deathtime`." Therefore, we'll only keep rows where LOS >= 0.

Note that we include patients who died in-hospital, whose `dischtime` is the same as `deathtime`.

In [94]:
df['length_of_stay_days'] = (df['dischtime'] - df['admittime']).dt.days
df = df[df['length_of_stay_days'] >= 0]
print(len(df))

220887


### in-hospital mortality

If applicable, `deathtime` provides the time of in-hospital death for the patient. Note that `deathtime` is only present if the patient died in-hospital, and is almost always the same as the patient’s `dischtime`. However, there can be some discrepancies due to typographical errors.

`deathtime` is the time of death of a patient if they died in hospital. If the patient did not die within the hospital for the given hospital admission, `deathtime` will be null.

Note that people discharged into hospice are not considered an in-hospital death.

`hospital_expire_flag` is a binary flag which indicates whether the patient died within the given hospitalization. 1 indicates death in the hospital, and 0 indicates survival to hospital discharge.

In [95]:
inconsistent_mask = ((df['deathtime'].isna()) & (df['hospital_expire_flag'] != 0)) | \
                    ((~df['deathtime'].isna()) & (df['hospital_expire_flag'] != 1))
len(df[inconsistent_mask])

8

Per the above, whenever there are inconsistencies between `deathtime` and `hospital_expire_flag`, `hospital_expire_flag` is the correct column (because `discharge_location` is consistent with `dod`). So we can use `hospital_expire_flag` as the label.

In [96]:
inhospital_deaths_df = df[df.hospital_expire_flag == 1]
subject_counts = inhospital_deaths_df.groupby('subject_id').size()
duplicated_subjects = list(subject_counts[subject_counts > 1].index)
duplicated_df = inhospital_deaths_df[inhospital_deaths_df['subject_id'].isin(duplicated_subjects)]
len(duplicated_subjects)

6

Per the above, there are six subjects with multiple recorded admissions in which they've supposedly died in-hospital. To handle these cases, we'll remove any of their admissions in which `dod` is a date before `dischtime`.

In [97]:
num_before = len(df)
# remove rows where subject_id is in duplicated_subjects AND dod is before dischtime
condition = df['subject_id'].isin(duplicated_subjects) & (df['dod'].dt.date < df['dischtime'].dt.date)
df = df[~condition]

num_after = len(df)
assert num_before - num_after == len(duplicated_subjects)

len(df)

220881

In fact, we'll remove all rows where `dod` is before `dischtime` since these are likely organ donor accounts.

In [98]:
df = df[~(df['dod'].dt.date < df['dischtime'].dt.date)]
len(df)

220826

### 30-day all-cause readmission

In [99]:
# takes about 8 minutes
def find_admission_within_30_days(row):
    if row.name % 100 == 0:
        print(f"Processing row {row.name}...")

    # filter for row subject's admissions
    subject_df = df[df['subject_id'] == row['subject_id']]

    # filter for admissions after the current discharge time
    future_admissions = subject_df[subject_df['admittime'] > row['dischtime']]

    # find admissions within 30 days, if any
    within_30_days = future_admissions[future_admissions['admittime'] <= row['dischtime'] + pd.Timedelta(days=30)]

    if within_30_days.empty:
        return 0
    else:
        return 1

df['adm_within_30_days'] = df.apply(find_admission_within_30_days, axis=1)

Processing row 0...
Processing row 100...
Processing row 200...
Processing row 300...
Processing row 400...
Processing row 500...
Processing row 600...
Processing row 700...
Processing row 800...
Processing row 900...
Processing row 1000...
Processing row 1100...
Processing row 1200...
Processing row 1300...
Processing row 1400...
Processing row 1500...
Processing row 1600...
Processing row 1700...
Processing row 1800...
Processing row 1900...
Processing row 2000...
Processing row 2100...
Processing row 2200...
Processing row 2300...
Processing row 2400...
Processing row 2500...
Processing row 2600...
Processing row 2700...
Processing row 2800...
Processing row 2900...
Processing row 3000...
Processing row 3100...
Processing row 3200...
Processing row 3300...
Processing row 3400...
Processing row 3500...
Processing row 3600...
Processing row 3700...
Processing row 3800...
Processing row 3900...
Processing row 4000...
Processing row 4100...
Processing row 4200...
Processing row 4300...


In theory, patients who die in-hospital should not have a readmission. However, there are 5 subjects who do seem to have a readmission. It looks like these are organ donor account cases. For any rows that have `hospital_expire_flag == 1` _and_ `adm_within_30_days == 1`, we'll remove the organ donor account row and set `adm_within_30_days == 0`.

In [100]:
len(df[(df['hospital_expire_flag'] == 1) & (df['adm_within_30_days'] == 1)])

5

In [113]:
hadm_id_remove = [27698265, 26524037, 27350110, 23614996, 24403793]
hadm_id_set_adm_within_30_days_to_0 = [20331070, 25862298, 24414758, 21122172, 25449052]

In [120]:
df = df[~df['hadm_id'].isin(hadm_id_remove)]
df.loc[df['hadm_id'].isin(hadm_id_set_adm_within_30_days_to_0), 'adm_within_30_days'] = 0
len(df)

220821

We'll save the df before creating splits.

In [121]:
df.to_csv("/gpfs/scratch/as16583/symile/symile/risk_prediction/datasets/df_before_splits.csv", index=False)

In [122]:
print("number of admissions: ", len(df))
print("number of admissions with both cxr and ecg: ", len(df[(~df['cxr_path'].isna()) & (~df['ecg_path'].isna())]))
print("number of admissions with only cxr: ", len(df[(~df['cxr_path'].isna()) & (df['ecg_path'].isna())]))
print("number of admissions with only ecg: ", len(df[(df['cxr_path'].isna()) & (~df['ecg_path'].isna())]))

number of admissions:  220821
number of admissions with both cxr and ecg:  53939
number of admissions with only cxr:  16291
number of admissions with only ecg:  150591


### create splits

In [ ]:
df = pd.read_csv("/gpfs/scratch/as16583/symile/symile/risk_prediction/datasets/df_before_splits.csv")

We'll now create dataset splits. We'll make sure that there is no overlap in subject_id between the splits.

In [128]:
unique_subject_ids = df["subject_id"].unique()
len(unique_subject_ids)

107386

In [129]:
train_size = .8
test_size = .5

train_subject_ids, val_subject_ids = train_test_split(unique_subject_ids, train_size=train_size, shuffle=True, random_state=42)
val_subject_ids, test_subject_ids = train_test_split(val_subject_ids, test_size=0.5, shuffle=True, random_state=42)

print("len(train_subject_ids): ", len(train_subject_ids))
print("len(val_subject_ids): ", len(val_subject_ids))
print("len(test_subject_ids): ", len(test_subject_ids))

len(train_subject_ids):  85908
len(val_subject_ids):  10739
len(test_subject_ids):  10739


In [130]:
train_df = df[df["subject_id"].isin(train_subject_ids)]
val_df = df[df["subject_id"].isin(val_subject_ids)]
test_df = df[df["subject_id"].isin(test_subject_ids)]
print("len(train_df): ", len(train_df))
print("len(val_df): ", len(val_df))
print("len(test_df): ", len(test_df))

len(train_df):  176518
len(val_df):  22273
len(test_df):  22030


In [131]:
assert set(train_df["subject_id"]).isdisjoint(set(val_df["subject_id"]))
assert set(train_df["subject_id"]).isdisjoint(set(test_df["subject_id"]))
assert set(val_df["subject_id"]).isdisjoint(set(test_df["hadm_id"]))

Now, we'll create the length of stay quantiles using the train set, and then apply them to the val and test sets.

In [133]:
train_df.length_of_stay_days.describe()

count    176518.000000
mean          4.261424
std           6.921329
min           0.000000
25%           1.000000
50%           2.000000
75%           5.000000
max         249.000000
Name: length_of_stay_days, dtype: float64

In [137]:
train_df['los_quantile'], train_bins = pd.qcut(train_df['length_of_stay_days'], q=4, labels=False, retbins=True, duplicates='raise')
train_bins

/tmp/ipykernel_411655/3982929101.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_df['los_quantile'], train_bins = pd.qcut(train_df['length_of_stay_days'], q=4, labels=False, retbins=True, duplicates='raise')


array([  0.,   1.,   2.,   5., 249.])

In [140]:
val_df['los_quantile'], val_bins = pd.cut(val_df['length_of_stay_days'], bins=train_bins, labels=False, retbins=True, include_lowest=True)
val_bins

/tmp/ipykernel_411655/3879431466.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  val_df['los_quantile'], val_bins = pd.cut(val_df['length_of_stay_days'], bins=train_bins, labels=False, retbins=True, include_lowest=True)


array([  0.,   1.,   2.,   5., 249.])

In [141]:
test_df['los_quantile'], test_bins = pd.cut(test_df['length_of_stay_days'], bins=train_bins, labels=False, retbins=True, include_lowest=True)
test_bins

/tmp/ipykernel_411655/3217424161.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_df['los_quantile'], test_bins = pd.cut(test_df['length_of_stay_days'], bins=train_bins, labels=False, retbins=True, include_lowest=True)


array([  0.,   1.,   2.,   5., 249.])

In [147]:
train_df.to_csv("/gpfs/scratch/as16583/symile/symile/risk_prediction/datasets/train_df.csv", index=False)
val_df.to_csv("/gpfs/scratch/as16583/symile/symile/risk_prediction/datasets/val_df.csv", index=False)
test_df.to_csv("/gpfs/scratch/as16583/symile/symile/risk_prediction/datasets/test_df.csv", index=False)